### Analytical Evaluation of the Effective Dynamics (FAPT)

While the numerical propagation of the effective state is efficient for single pulse sequences, large parameter sweeps and pulse optimizations require a less computationally demanding approach. To achieve this, the perturbative construction can be evaluated analytically.

Instead of performing the time-evolution numerically over many small steps $\delta t$, this approach uses closed-form symbolic expressions for both the transformation matrix $W(\boldsymbol\lambda)$ and the effective Hamiltonian $H_{\mathrm{eff,ad}}$. The time evolution operator $U(t, t_i)$ over the entire pulse duration is then approximated using the **Magnus expansion** (up to second order), effectively evaluating the evolution in a single analytical step. 

The following code tests this analytical framework by substituting symbolic pulse shapes (like a Gaussian amplitude) into the analytically derived Magnus operator.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import sympy as sp
from utils import * 
from FAPT import * 
from pulse_optimization import * 
s = Simulation()
rH = 3
rW = 2

In [2]:
from pathlib import Path
import numpy as np
import sympy as sp
DRIVE_DIRECTORY = Path("operational_res/analytical_drive_data")

def load_analytical_drive(pulse_type, drive_directory=DRIVE_DIRECTORY):
    file_path = Path(drive_directory) / f"analytical_drive_{pulse_type}.npz"

    if not file_path.exists():
        raise FileNotFoundError(f"Die Datei '{file_path}' wurde nicht gefunden.")

    with np.load(file_path, allow_pickle=False) as saved_data:
        stored_pulse_type = str(saved_data["pulse_type"].item())
        Ar_expression = sp.sympify(saved_data["Ar_srepr"].item())
        Ai_expression = sp.sympify(saved_data["Ai_srepr"].item())
        dwd_expression = sp.sympify(saved_data["dwd_srepr"].item())

        wd0_serialized = saved_data["wd0_srepr"].item()
        wd0_expression = sp.sympify(wd0_serialized) if wd0_serialized else None
    if stored_pulse_type != pulse_type:
        raise ValueError(f"Angefordert wurde '{pulse_type}', die Datei enthält jedoch '{stored_pulse_type}'.")

    pulse_data = {
        "Ar": Ar_expression,
        "Ai": Ai_expression,
        "dwd": dwd_expression,
        "wd0": wd0_expression,
    }
    print(f"Symbolische Pulsform geladen: '{file_path}'")
    return pulse_data

gauss_drive = load_analytical_drive(pulse_type="gauss")
Ar_gauss_expr = gauss_drive["Ar"]
Ai_gauss_expr = gauss_drive["Ai"]
dwd_gauss_expr = gauss_drive["dwd"]
wd0_gauss_expr = gauss_drive["wd0"]
tanh_drive = load_analytical_drive(pulse_type="tanh")
Ar_tanh_expr = tanh_drive["Ar"]
Ai_tanh_expr = tanh_drive["Ai"]
dwd_tanh_expr = tanh_drive["dwd"]
wd0_tanh_expr = tanh_drive["wd0"]

def get_symbol_by_name(expressions, symbol_name):
    free_symbols = set().union(*(expression.free_symbols for expression in expressions if expression is not None))
    matching_symbols = [symbol for symbol in free_symbols if symbol.name == symbol_name]

    if not matching_symbols:
        raise KeyError(f"Das Symbol '{symbol_name}' kommt in den geladenen Ausdrücken nicht vor.")

    return matching_symbols[0]

loaded_expressions = (Ar_gauss_expr, Ai_gauss_expr, dwd_gauss_expr)
t_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name="t")
sigma_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name=r"\sigma")
tg_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name="t_g")
Ar_gauss_expr

Symbolische Pulsform geladen: 'operational_res/analytical_drive_data/analytical_drive_gauss.npz'
Symbolische Pulsform geladen: 'operational_res/analytical_drive_data/analytical_drive_tanh.npz'


47.0783429077837*pi*(exp(1/(8*\sigma**2)) - exp((2*t - t_g)**2/(8*\sigma**2*t_g**2)))*exp(-(2*t - t_g)**2/(8*\sigma**2*t_g**2))/(t_g*(sqrt(2)*sqrt(pi)*\sigma*exp(1/(8*\sigma**2))*erf(sqrt(2)/(4*\sigma)) - 1))

In [3]:
def replace_loaded_symbols(expression, t_sym, sigma_sym, tg_sym):
    symbol_map = {}

    for symbol in expression.free_symbols:
        if symbol.name == "t":
            symbol_map[symbol] = t_sym
        elif symbol.name in ("sigma", "sigma_r", r"\sigma"):
            symbol_map[symbol] = sigma_sym
        elif symbol.name in ("tg", "t_g", r"t_{g}"):
            symbol_map[symbol] = tg_sym

    return expression.xreplace(symbol_map)


def make_analytical_pulse_builder(Ar_expr, Ai_expr, dwd_expr):
    def pulse_shape_builder(t_sym, p_syms):
        tg_sym, sigma_sym, wd_base_sym, wd_offset_sym, amp_scale_sym = p_syms

        Ar = replace_loaded_symbols(expression=Ar_expr, t_sym=t_sym, sigma_sym=sigma_sym, tg_sym=tg_sym)
        Ai = replace_loaded_symbols(expression=Ai_expr, t_sym=t_sym, sigma_sym=sigma_sym, tg_sym=tg_sym)
        dwd = replace_loaded_symbols(expression=dwd_expr, t_sym=t_sym, sigma_sym=sigma_sym, tg_sym=tg_sym)

        A = amp_scale_sym * (Ar + sy.I * Ai)
        wd = wd_base_sym + dwd + wd_offset_sym

        return A, wd

    return pulse_shape_builder

In [4]:
# ==============================================================================
# EINMALIGE SYMBOLISCHE FLOQUET-VORBEREITUNG
# ==============================================================================

gauss_drive = load_analytical_drive(pulse_type="gauss")

gauss_pulse_builder = make_analytical_pulse_builder(Ar_expr=gauss_drive["Ar"], Ai_expr=gauss_drive["Ai"], dwd_expr=gauss_drive["dwd"])

base_parameter_names = [
    "t_g",
    "sigma",
    "wd_base",
    "wd_offset",
    "amp_scale",
]

H_gauss_base, M_gauss_base, M_inv_0_gauss_base = prepare_floquet_functions(pulse_shape_builder=gauss_pulse_builder, p_names=base_parameter_names, s=s, rH=rH, rW=rW, include_geometric=False, include_micromotion=False, include_g_correction=False, verbose=False)

Symbolische Pulsform geladen: 'operational_res/analytical_drive_data/analytical_drive_gauss.npz'


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sy

sigma_fixed = 0.3
wd0_fixed = float(wd0_gauss_expr)
tg_values = np.linspace(100.0, 1000.0, 50)
pts_per_ns = 40.0

global_param_ranges = {
    "wd_offset": (-0.01, 0.01),
    "amp_scale": (0.8, 1.2),
}

param_ranges = global_param_ranges.copy()
param_names = list(param_ranges.keys())

Ar_direct_func = sy.lambdify((t_sym, sigma_sym, tg_sym), Ar_gauss_expr, modules=["numpy"])
Ai_direct_func = sy.lambdify((t_sym, sigma_sym, tg_sym), Ai_gauss_expr, modules=["numpy"])
dwd_direct_func = sy.lambdify((t_sym, sigma_sym, tg_sym), dwd_gauss_expr, modules=["numpy"])

results = []
x0 = np.asarray([0.0, 1.0])

for tg_target in tg_values:
    def make_functions(tg):
        def H_func(t, *p_vals):
            return H_gauss_base(t, tg, sigma_fixed, wd0_fixed, *p_vals)

        def M_func(t, *p_vals):
            return M_gauss_base(t, tg, sigma_fixed, wd0_fixed, *p_vals)

        def M_inv_0_func(*p_vals):
            return M_inv_0_gauss_base(tg, sigma_fixed, wd0_fixed, *p_vals)

        H_func.is_time_dependent = getattr(H_gauss_base, "is_time_dependent", True)

        return H_func, M_func, M_inv_0_func

    H_func, M_func, M_inv_0_func = make_functions(tg_target)

    best_params, floq_pop = optimize_pulse_parameters(H_func=H_func, M_func=M_func, M_inv_0_func=M_inv_0_func, param_ranges=param_ranges, s=s, tg=tg_target, N_t=160, target_metric="population", U_target=s.U_ideal, verbose=False, popsize=10, maxiter=20, tol=1e-8, polish=True, x0=x0)

    p_vals = tuple(best_params[name] for name in param_names)
    wd = wd0_fixed + best_params["wd_offset"]

    tg_final, floq_proc, U_corrected, U_ideal_corrected, global_phase = refine_gate_time_phase_aligned(p_vals=p_vals, tg_target=tg_target, wd=wd, H_func=H_func, M_func=M_func, M_inv_0_func=M_inv_0_func, s=s, phase_index=(0, 1), spectator_index=2, steps_per_ns=2.0, grid_points=31, function_factory=make_functions)

    def compute_direct_propagator(tg):
        def q_sig(t, args=None):
            Ar = Ar_direct_func(t, sigma_fixed, tg)
            Ai = Ai_direct_func(t, sigma_fixed, tg)
            dwd = dwd_direct_func(t, sigma_fixed, tg)
            phase = (wd0_fixed + best_params["wd_offset"] + dwd) * t

            return best_params["amp_scale"] * np.real(Ar * np.cos(phase) + Ai * np.sin(phase))

        n_pts_mes = max(101, int(np.ceil(tg * pts_per_ns)))
        t_mes = np.linspace(0.0, tg, n_pts_mes)
        U_mes = np.zeros((3, 3), dtype=complex)

        for j in range(3):
            amplitudes = resonant_subspace_column_evolution([s.H0, [s.V1, q_sig]], t_mes, j, debug=False, s=s)
            U_mes[:, j] = amplitudes[:, -1]

        return U_mes

    U_mes_pop = compute_direct_propagator(tg_target)
    mes_pop = compute_gate_infidelity(U_mes_pop, s.U_ideal, metric="population", swap_indices=(0, 1))

    U_mes_proc = compute_direct_propagator(tg_final)
    mes_proc, U_mes_corrected, U_ideal_mes_corrected, _ = compute_phase_aligned_process_infidelity(U_mes_proc, s.U_ideal, phase_index=(0, 1), spectator_index=2)

    results.append({
        "tg": tg_target,
        "tg_final": tg_final,
        "floq_pop": floq_pop,
        "mes_pop": mes_pop,
        "floq_proc": floq_proc,
        "mes_proc": mes_proc,
        "wd_offset": best_params["wd_offset"],
        "amp_scale": best_params["amp_scale"],
    })

    x0 = np.asarray([best_params[name] for name in param_names])

    wd_center = best_params["wd_offset"]
    amp_center = best_params["amp_scale"]

    param_ranges = {
        "wd_offset": (max(global_param_ranges["wd_offset"][0], wd_center - 0.002), min(global_param_ranges["wd_offset"][1], wd_center + 0.002)),
        "amp_scale": (max(global_param_ranges["amp_scale"][0], amp_center - 0.04), min(global_param_ranges["amp_scale"][1], amp_center + 0.04)),
    }

    print(
        f"tg={tg_target:6.1f} ns | "
        f"tg_final={tg_final:10.6f} ns | "
        f"I_pop Floq/Direct={floq_pop:.3e}/{mes_pop:.3e} | "
        f"I_proc Floq/Direct={floq_proc:.3e}/{mes_proc:.3e} | "
        f"dwd={best_params['wd_offset']:+.6e} | "
        f"amp_scale={best_params['amp_scale']:.6f}"
    )

tg= 100.0 ns | tg_final= 98.984707 ns | I_pop Floq/Direct=2.889e-02/5.786e-01 | I_proc Floq/Direct=2.130e-02/6.717e-01 | dwd=+5.310100e-03 | amp_scale=1.200000
tg= 118.4 ns | tg_final=117.357600 ns | I_pop Floq/Direct=2.109e-03/3.773e-01 | I_proc Floq/Direct=1.770e-03/7.978e-01 | dwd=+5.679003e-03 | amp_scale=1.199997
tg= 136.7 ns | tg_final=135.799974 ns | I_pop Floq/Direct=9.587e-06/2.459e-01 | I_proc Floq/Direct=2.189e-06/8.747e-01 | dwd=+3.679129e-03 | amp_scale=1.162485
tg= 155.1 ns | tg_final=154.250583 ns | I_pop Floq/Direct=1.448e-04/1.726e-01 | I_proc Floq/Direct=6.838e-05/8.866e-01 | dwd=+1.921651e-03 | amp_scale=1.122485
tg= 173.5 ns | tg_final=172.694829 ns | I_pop Floq/Direct=8.260e-08/1.325e-01 | I_proc Floq/Direct=9.935e-07/8.526e-01 | dwd=+7.033951e-04 | amp_scale=1.083728
tg= 191.8 ns | tg_final=191.114293 ns | I_pop Floq/Direct=8.274e-08/1.022e-01 | I_proc Floq/Direct=5.648e-07/7.966e-01 | dwd=+2.957671e-04 | amp_scale=1.065663
tg= 210.2 ns | tg_final=209.527773 ns | 

In [6]:
from pathlib import Path
import json

save_path = Path("operational_res/analytical_pulse_results.json")
save_path.parent.mkdir(parents=True, exist_ok=True)

with save_path.open("w", encoding="utf-8") as file:
    json.dump(results, file, indent=2, default=lambda value: value.item())

print(f"Ergebnisse gespeichert: {save_path}")

Ergebnisse gespeichert: operational_res/analytical_pulse_results.json
